# Advanced Problems with Solutions: Python Attribute Read Accessors

This notebook is a practice-focused expansion of the source topic **Attribute Read Accessors**:
`__getattr__`, `__getattribute__`, safe delegation, recursion hazards, class-level access via
metaclasses, and method lookup.

## How to use this notebook

For each problem:

1. Read the **Problem** and predict the behavior before running anything.
2. Edit/run the **Starter** cell if you want to solve it yourself.
3. Compare with the **Solution**.
4. Run the **Verification** assertions.
5. Modify the solution and add your own edge cases.

## Best-practice rules used throughout

- Prefer `__getattr__` when you only need a fallback for **missing** attributes.
- Override `__getattribute__` only when you truly need to intercept **every** attribute read.
- Inside `__getattribute__`, delegate with `object.__getattribute__(self, name)` (or carefully with
  `super().__getattribute__(name)`) to avoid recursion.
- If a missing attribute really is missing, raise `AttributeError`, not `KeyError` or `ValueError`.
- Remember that `hasattr(obj, "x")` works by catching `AttributeError`.
- Avoid pretending underscore-blocking is a security boundary; Python introspection can bypass it.
- Preserve normal descriptor/property/method semantics unless you intentionally replace them.
- When forwarding/proxying, decide precedence explicitly: proxy attributes first, wrapped object second,
  or vice versa.
- Keep hooks small, deterministic, and well-tested—they affect a huge fraction of object behavior.

## Reference: mental model of a normal attribute read

For `obj.name`, the useful high-level model is:

1. `type(obj).__getattribute__(obj, "name")` handles the lookup.
2. Normal lookup considers descriptors, the instance namespace, class attributes, inheritance, etc.
3. If normal lookup ultimately raises `AttributeError`, Python may call `type(obj).__getattr__(obj, "name")`.
4. If `__getattr__` cannot provide a value, it should also raise `AttributeError`.

This is simplified, but it is the right model for the exercises below.

In [1]:
# Small helper used by several verification cells.
def assert_raises(exc_type, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc_type as exc:
        return exc
    else:
        raise AssertionError(f"Expected {exc_type.__name__}")

# Problem 1 — Predict the lookup path

Consider a class with a regular instance attribute, a class attribute, a property, and `__getattr__`.

**Task:** predict which accesses invoke `__getattr__`. Then verify your prediction.

Questions:

- Does `__getattr__` run for an existing instance attribute?
- Does it run for a class attribute?
- Does it run for a property?
- Does it run after a property raises `AttributeError`?
- Does it run for an actually missing name?

In [2]:
# Starter: make your predictions before running the solution.
predictions = {
    "instance": None,
    "class": None,
    "property": None,
    "property_raises_attribute_error": None,
    "missing": None,
}

## Solution 1

`__getattr__` is a fallback. It is not the first step for ordinary reads.
A subtle case is a property whose getter raises `AttributeError`: that failure is treated as a failed
attribute lookup, so `__getattr__` can be invoked for the same name.

In [3]:
class LookupDemo:
    class_value = "from class"

    def __init__(self):
        self.instance_value = "from instance"
        self.events = []

    @property
    def property_value(self):
        return "from property"

    @property
    def fragile_property(self):
        raise AttributeError("property could not produce a value")

    def __getattr__(self, name):
        # Use an existing attribute here. Since events exists, this is safe.
        self.events.append(name)
        return f"fallback:{name}"


d = LookupDemo()
assert d.instance_value == "from instance"
assert d.class_value == "from class"
assert d.property_value == "from property"
assert d.fragile_property == "fallback:fragile_property"
assert d.missing == "fallback:missing"

assert d.events == ["fragile_property", "missing"]
d.events

['fragile_property', 'missing']

# Problem 2 — Private-name fallback without infinite recursion

Implement behavior like this:

- `obj.age` should fall back to `obj._age` if `age` is missing.
- If neither exists, raise `AttributeError`.
- Do **not** recursively call your own `__getattr__` forever.

A common broken implementation is:

```python
def __getattr__(self, name):
    alt = "_" + name
    return getattr(self, alt)   # danger if alt is also missing
```

**Task:** write a safe version.

In [4]:
# Starter
class PrivateFallbackStarter:
    def __getattr__(self, name):
        # TODO: safely look up "_" + name without recursively re-entering this method.
        raise AttributeError(name)

## Solution 2

Delegate the alternate-name lookup to the normal implementation via
`object.__getattribute__(self, alt_name)`. If that also fails, raise a clear `AttributeError`.

In [5]:
class PrivateFallback:
    def __init__(self, age):
        self._age = age

    def __getattr__(self, name):
        alt_name = "_" + name
        try:
            return object.__getattribute__(self, alt_name)
        except AttributeError:
            raise AttributeError(
                f"{type(self).__name__!s} has no attribute {name!r} "
                f"and no fallback {alt_name!r}"
            ) from None


p = PrivateFallback(42)
assert p.age == 42
assert p._age == 42

exc = assert_raises(AttributeError, getattr, p, "salary")
assert "salary" in str(exc)
assert "_salary" in str(exc)

# Problem 3 — Lazy computed attributes with caching

Build an object with an expensive value named `report`.

Requirements:

- `report` should not be computed in `__init__`.
- The first `obj.report` access computes it.
- Later accesses reuse the cached result.
- Unrelated missing attributes must still raise `AttributeError`.
- Avoid `getattr(self, ...)` patterns that can accidentally recurse.

In [6]:
# Starter
class LazyReportStarter:
    def __init__(self, raw):
        self.raw = raw

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 3

`__getattr__` is a natural fit for lazy values because it only runs while the name is absent.
After computing the value, store it under the requested name in `self.__dict__`. Future reads then
succeed normally without calling `__getattr__`.

In [7]:
class LazyReport:
    def __init__(self, raw):
        self.raw = tuple(raw)
        self.compute_count = 0

    def _build_report(self):
        self.compute_count += 1
        return {
            "count": len(self.raw),
            "sum": sum(self.raw),
            "min": min(self.raw),
            "max": max(self.raw),
        }

    def __getattr__(self, name):
        if name == "report":
            value = self._build_report()
            self.__dict__["report"] = value
            return value
        raise AttributeError(f"{type(self).__name__} has no attribute {name!r}")


obj = LazyReport([5, 2, 9, 4])
assert obj.compute_count == 0
first = obj.report
assert obj.compute_count == 1
second = obj.report
assert obj.compute_count == 1
assert first is second
assert first["sum"] == 20
assert_raises(AttributeError, getattr, obj, "does_not_exist")

AttributeError("LazyReport has no attribute 'does_not_exist'")

# Problem 4 — Default-on-missing attributes without shared mutable state

Create an object where every missing public attribute is initialized from a factory.

Example:

```python
bag.notes.append("A")
bag.tags.add("python")
```

Requirements:

- A separate default object must be created per missing attribute.
- Different instances must not share defaults.
- Private/dunder names should not be silently invented.
- Once created, the attribute should behave like a normal instance attribute.

In [8]:
# Starter
class DefaultFactoryStarter:
    def __init__(self, factory):
        self._factory = factory

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 4

Store the factory normally, then call it only for eligible missing public names. Cache the created
value in the instance dictionary.

In [9]:
class DefaultFactoryObject:
    def __init__(self, factory):
        self._factory = factory

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)

        factory = object.__getattribute__(self, "_factory")
        value = factory()
        self.__dict__[name] = value
        return value


a = DefaultFactoryObject(list)
b = DefaultFactoryObject(list)

a.notes.append("A")
a.notes.append("B")
b.notes.append("other instance")

assert a.notes == ["A", "B"]
assert b.notes == ["other instance"]
assert a.notes is not b.notes
assert_raises(AttributeError, getattr, a, "_internal_missing")

AttributeError('_internal_missing')

# Problem 5 — Logging missing attributes while preserving Python semantics

You want diagnostics whenever code requests an unknown attribute, but you **do not** want to invent a
fallback value.

Requirements:

- Log/record the missing name.
- Raise `AttributeError` afterward.
- Make `hasattr(obj, "missing")` return `False`.
- Existing attributes must remain unaffected.

In [10]:
# Starter
class MissingAuditStarter:
    def __init__(self):
        self.audit = []

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 5

The key best practice is to raise `AttributeError` after recording the event. Returning `None` would
make a typo look like a valid attribute and would make `hasattr` return `True`.

In [11]:
class MissingAudit:
    def __init__(self):
        self.present = 10
        self.audit = []

    def __getattr__(self, name):
        self.audit.append(name)
        raise AttributeError(f"{type(self).__name__} has no attribute {name!r}")


m = MissingAudit()
assert m.present == 10
assert hasattr(m, "present") is True
assert hasattr(m, "missing") is False
assert m.audit == ["missing"]

assert_raises(AttributeError, getattr, m, "another")
assert m.audit == ["missing", "another"]

# Problem 6 — Mapping-backed dynamic attributes with explicit precedence

Implement a record backed by a dictionary:

```python
record = MappingView({"name": "Ada", "items": 7})
record.name   # -> "Ada"
```

Requirements:

- Real methods and attributes of the wrapper must win over mapping keys.
- Mapping keys are only a fallback.
- Missing keys must become `AttributeError`, not `KeyError`.
- A key named `keys` must not replace the wrapper's own `keys()` method.

In [12]:
# Starter
class MappingViewStarter:
    def __init__(self, data):
        self._data = dict(data)

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 6

This is exactly the kind of behavior `__getattr__` handles well: normal lookup gets first chance, and
the mapping is consulted only if normal lookup fails.

In [13]:
class MappingView:
    def __init__(self, data):
        self._data = dict(data)

    def keys(self):
        return tuple(self._data.keys())

    def __getattr__(self, name):
        data = object.__getattribute__(self, "_data")
        try:
            return data[name]
        except KeyError:
            raise AttributeError(
                f"{type(self).__name__} has no attribute or mapped key {name!r}"
            ) from None


record = MappingView({"name": "Ada", "items": 7, "keys": "data-key-called-keys"})
assert record.name == "Ada"
assert record.items == 7
assert callable(record.keys)
assert "name" in record.keys()
assert_raises(AttributeError, getattr, record, "email")

AttributeError("MappingView has no attribute or mapped key 'email'")

# Problem 7 — Deprecated attribute aliases + discoverability

Suppose an API renamed `old_name` to `new_name`.

Requirements:

- Existing code reading `obj.old_name` should still work for now.
- Emit a `DeprecationWarning`.
- `obj.new_name` remains the canonical attribute.
- `dir(obj)` should include `old_name` so interactive users can discover the compatibility alias.
- Unknown names still raise `AttributeError`.

In [14]:
# Starter
# Hint: combine __getattr__ with __dir__.

## Solution 7

`__getattr__` can implement compatibility aliases. `__dir__` is optional but improves introspection
when you intentionally expose dynamic names.

In [15]:
import warnings

class RenamedAPI:
    def __init__(self, value):
        self.new_name = value

    def __getattr__(self, name):
        if name == "old_name":
            warnings.warn(
                "'old_name' is deprecated; use 'new_name'",
                DeprecationWarning,
                stacklevel=2,
            )
            return self.new_name
        raise AttributeError(name)

    def __dir__(self):
        return sorted(set(super().__dir__()) | {"old_name"})


api = RenamedAPI(123)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert api.old_name == 123
    assert len(caught) == 1
    assert issubclass(caught[0].category, DeprecationWarning)

assert api.new_name == 123
assert "old_name" in dir(api)
assert_raises(AttributeError, getattr, api, "totally_unknown")

AttributeError('totally_unknown')

# Problem 8 — Instrument every attribute read safely

Now you really do need `__getattribute__`: record reads of public attributes.

Requirements:

- Every public attribute read should increment a counter.
- The counter machinery itself must not recurse.
- Private names beginning with `_` should not be counted.
- Methods are attributes too; reading a method should count.
- Behavior of the actual attributes should remain normal.

**Challenge:** implement this without accessing `self._read_counts` through your custom hook.

In [16]:
# Starter
class ReadCounterStarter:
    def __init__(self):
        self.value = 10
        self._read_counts = {}

    def __getattribute__(self, name):
        # TODO: be very careful here.
        return object.__getattribute__(self, name)

## Solution 8

Inside `__getattribute__`, use `object.__getattribute__` for internal state. Do the minimum work
possible, then delegate the requested attribute to the normal implementation.

In [17]:
class ReadCounter:
    def __init__(self):
        self.value = 10
        self._read_counts = {}

    def double(self):
        return self.value * 2

    def __getattribute__(self, name):
        if not name.startswith("_"):
            counts = object.__getattribute__(self, "_read_counts")
            counts[name] = counts.get(name, 0) + 1
        return object.__getattribute__(self, name)

    def counts(self):
        return dict(object.__getattribute__(self, "_read_counts"))


rc = ReadCounter()
assert rc.value == 10              # counts "value"
method = rc.double                 # counts "double"
assert method() == 20              # inside double, reading self.value counts again
snapshot = rc.counts()             # reading "counts" counts as well

assert snapshot["value"] == 2
assert snapshot["double"] == 1
assert snapshot["counts"] == 1

# Problem 9 — Selectively blocking underscore-prefixed reads

Create a class that rejects direct reads of names beginning with a **single underscore** while still
allowing a public property to expose the value.

Requirements:

- `obj.name` works through a property.
- `obj._name` raises `AttributeError`.
- Dunder attributes such as `obj.__class__` continue to work.
- The property itself must be able to read `_name` without triggering your block.
- Include a note about why this is **not** a security mechanism.

In [18]:
# Starter
class HiddenReadStarter:
    def __init__(self, name):
        self._name = name

    def __getattribute__(self, name):
        # TODO
        return object.__getattribute__(self, name)

## Solution 9

The public property bypasses the custom hook by calling `object.__getattribute__(self, "_name")`.

This can enforce an API convention, but it is **not a security boundary**. Python code can still use
`object.__getattribute__(obj, "_name")`, inspect state, or otherwise bypass the convention.

In [19]:
class HiddenRead:
    def __init__(self, name):
        self._name = name

    def __getattribute__(self, name):
        if name.startswith("_") and not name.startswith("__"):
            raise AttributeError(f"private read blocked: {name}")
        return object.__getattribute__(self, name)

    @property
    def name(self):
        return object.__getattribute__(self, "_name")


h = HiddenRead("Ada")
assert h.name == "Ada"
assert h.__class__ is HiddenRead
assert_raises(AttributeError, getattr, h, "_name")

# Demonstrates why the block is only an API convention:
assert object.__getattribute__(h, "_name") == "Ada"

# Problem 10 — A safe transparent-ish proxy

Build a proxy around another object.

Requirements:

- Attributes defined on the proxy itself must win.
- Missing proxy attributes should be delegated to the wrapped target.
- Missing attributes on the target should still raise `AttributeError`.
- The wrapper reference must be read safely.
- Methods retrieved from the target should remain correctly bound to the target.

In [20]:
# Starter
class ProxyStarter:
    def __init__(self, target):
        self._target = target

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 10

Using `__getattr__` means normal proxy attributes are resolved first. Only missing names are forwarded.
`getattr(target, name)` is appropriate here because we intentionally want the target's complete normal
attribute semantics.

In [21]:
class Service:
    def __init__(self, base):
        self.base = base

    def add(self, x):
        return self.base + x


class Proxy:
    def __init__(self, target, label):
        self._target = target
        self.label = label

    def describe_proxy(self):
        return f"Proxy<{self.label}>"

    def __getattr__(self, name):
        target = object.__getattribute__(self, "_target")
        try:
            return getattr(target, name)
        except AttributeError:
            raise AttributeError(
                f"{type(self).__name__} and wrapped {type(target).__name__} "
                f"have no attribute {name!r}"
            ) from None


service = Service(10)
proxy = Proxy(service, "primary")

assert proxy.label == "primary"             # proxy attribute wins
assert proxy.base == 10                     # delegated data attribute
bound = proxy.add                           # delegated bound method
assert bound(5) == 15
assert getattr(bound, "__self__", None) is service
assert_raises(AttributeError, getattr, proxy, "missing")

AttributeError("Proxy and wrapped Service have no attribute 'missing'")

# Problem 11 — Method access is attribute access

Instrument method retrieval and distinguish it from method execution.

Requirements:

- Log when `obj.work` is **read**.
- Do not falsely claim that `__getattribute__` itself executed the method.
- Show that grabbing the bound method without calling it still triggers attribute access.

In [22]:
# Starter: predict the event order before running.

## Solution 11

`obj.work()` conceptually performs an attribute read to obtain `work`, then calls the returned object.
The lookup hook sees the read; the method body sees the call.

In [23]:
class MethodTrace:
    def __init__(self):
        self._events = []

    def __getattribute__(self, name):
        if name == "work":
            events = object.__getattribute__(self, "_events")
            events.append("read work")
        return object.__getattribute__(self, name)

    def work(self, x):
        events = object.__getattribute__(self, "_events")
        events.append(f"execute work({x})")
        return x * 2

    def events(self):
        return tuple(object.__getattribute__(self, "_events"))


mt = MethodTrace()
f = mt.work
assert mt.events() == ("read work",)

assert f(6) == 12
assert mt.events() == ("read work", "execute work(6)")

assert mt.work(7) == 14
assert mt.events()[-2:] == ("read work", "execute work(7)")

# Problem 12 — Descriptor precedence and why naive dictionary lookup is wrong

A major reason to delegate to Python's normal lookup machinery is that attributes are not merely
entries in `self.__dict__`.

Build a data descriptor named `score` whose `__get__` returns a controlled value. Then deliberately
place a conflicting `"score"` key in the instance dictionary.

**Task:** verify which value wins and explain why reaching directly into `__dict__` is not a general
replacement for normal attribute lookup.

In [24]:
# Starter
class DescriptorStarter:
    def __get__(self, instance, owner):
        pass

    def __set__(self, instance, value):
        pass

## Solution 12

A **data descriptor** (one defining `__set__` or `__delete__`, and normally `__get__`) has higher
precedence than the instance dictionary during normal lookup.

In [25]:
class NonNegative:
    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError("must be non-negative")
        instance.__dict__[self.storage_name] = value


class Player:
    score = NonNegative()

    def __init__(self, score):
        self.score = score


player = Player(10)

# Create a conflicting ordinary key manually.
player.__dict__["score"] = 999

assert player.__dict__["score"] == 999
assert player.score == 10  # data descriptor wins over instance __dict__["score"]

player.score = 15
assert player.score == 15
assert player.__dict__["_score"] == 15

# Problem 13 — `hasattr` only suppresses `AttributeError`

Create two dynamic properties:

- `optional` should behave as absent and make `hasattr(obj, "optional")` return `False`.
- `broken` should raise `RuntimeError` to indicate a real internal failure.

**Task:** verify that `hasattr` suppresses the first case but does not hide the second.

In [26]:
# Starter
class HasattrStarter:
    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

## Solution 13

This is why `AttributeError` should mean "the attribute is unavailable", while unrelated failures
should keep their real exception type.

In [27]:
class HasattrSemantics:
    def __getattr__(self, name):
        if name == "optional":
            raise AttributeError(name)
        if name == "broken":
            raise RuntimeError("backend failed")
        raise AttributeError(name)


hs = HasattrSemantics()
assert hasattr(hs, "optional") is False

exc = assert_raises(RuntimeError, hasattr, hs, "broken")
assert "backend failed" in str(exc)

# Problem 14 — Class-level fallback attributes with a metaclass

Instance `__getattr__` does not control missing attributes read from the class itself.

Implement:

```python
FeatureFlags.BETA
```

so unknown **uppercase** class attributes are read from a registry.

Requirements:

- Normal class attributes still win.
- Uppercase names may fall back to a registry.
- Lowercase unknown names raise `AttributeError`.
- Do not add instance-level magic for this problem.

In [28]:
# Starter
class FlagMetaStarter(type):
    def __getattr__(cls, name):
        # TODO
        raise AttributeError(name)

## Solution 14

Classes are instances of their metaclass. Therefore, class-level read hooks belong on the metaclass.

In [29]:
class FlagMeta(type):
    _registry = {
        "BETA": True,
        "MAX_USERS": 250,
    }

    def __getattr__(cls, name):
        if name.isupper():
            registry = type.__getattribute__(cls, "_registry")
            try:
                return registry[name]
            except KeyError:
                pass
        raise AttributeError(f"{cls.__name__} has no class attribute {name!r}")


class FeatureFlags(metaclass=FlagMeta):
    STABLE = "built-in"


assert FeatureFlags.STABLE == "built-in"
assert FeatureFlags.BETA is True
assert FeatureFlags.MAX_USERS == 250
assert_raises(AttributeError, getattr, FeatureFlags, "beta")
assert_raises(AttributeError, getattr, FeatureFlags, "UNKNOWN_FLAG")

AttributeError("FeatureFlags has no class attribute 'UNKNOWN_FLAG'")

# Problem 15 — Intercept every class attribute read safely

Now override `__getattribute__` on a metaclass to audit public class-level reads.

Requirements:

- Audit reads such as `Settings.VERSION`.
- Avoid recursion while appending to the audit list.
- Preserve ordinary metaclass/type lookup.
- Do not audit dunder reads.

In [30]:
# Starter
class AuditMetaStarter(type):
    def __getattribute__(cls, name):
        # TODO
        return type.__getattribute__(cls, name)

## Solution 15

The same recursion rule applies at the metaclass level: use `type.__getattribute__(cls, ...)` for
internal access and for final delegation.

In [31]:
class AuditMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace["_class_reads"] = []
        return super().__new__(mcls, name, bases, namespace)

    def __getattribute__(cls, name):
        if not name.startswith("__") and name != "_class_reads":
            reads = type.__getattribute__(cls, "_class_reads")
            reads.append(name)
        return type.__getattribute__(cls, name)


class Settings(metaclass=AuditMeta):
    VERSION = "2.0"
    MODE = "safe"


assert Settings.VERSION == "2.0"
assert Settings.MODE == "safe"
reads = type.__getattribute__(Settings, "_class_reads")
assert reads == ["VERSION", "MODE"]

# Problem 16 — Special-method lookup is different

Predict what happens here:

```python
obj.__len__       # explicit attribute read
len(obj)          # implicit special-method operation
```

**Task:** show that explicit `obj.__len__` is an ordinary attribute read, while many implicit special
method operations are resolved specially by Python, typically on the type rather than by performing
the same ordinary instance lookup path.

This is an important limitation when trying to build "transparent" proxies with only `__getattr__`.

In [32]:
# Starter: predict how many '__len__' reads the custom hook will record.

## Solution 16

The explicit read goes through the instance's `__getattribute__`. The implicit `len(obj)` special
operation looks up the special method using Python's special-method machinery.

In [33]:
class SpecialMethodTrace:
    def __init__(self):
        self._reads = []

    def __getattribute__(self, name):
        if name == "__len__":
            reads = object.__getattribute__(self, "_reads")
            reads.append("explicit __len__ attribute read")
        return object.__getattribute__(self, name)

    def __len__(self):
        return 3

    def reads(self):
        return tuple(object.__getattribute__(self, "_reads"))


smt = SpecialMethodTrace()

explicit = smt.__len__
assert smt.reads() == ("explicit __len__ attribute read",)
assert explicit() == 3

# This should not add another ordinary '__len__' read to our instance hook.
assert len(smt) == 3
assert smt.reads() == ("explicit __len__ attribute read",)

# Problem 17 — Dynamic computed fields with dependency safety

Create an object supporting dynamic attributes named `total`, `mean`, and `range`.

Requirements:

- Base data is stored under a private name.
- Computed names are supplied by `__getattr__`.
- Results are cached after the first computation.
- If data changes through `replace_data`, the cache must be invalidated.
- Missing names raise `AttributeError`.
- Empty data should produce a meaningful `ValueError` for computations that need elements.

In [34]:
# Starter
class StatsStarter:
    def __init__(self, data):
        self._data = tuple(data)

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

    def replace_data(self, data):
        # TODO
        pass

## Solution 17

This combines fallback lookup, caching, and explicit invalidation. Note that a real computation failure
uses `ValueError`, not `AttributeError`, because the attribute exists conceptually but cannot be
computed from the current data.

In [35]:
class Stats:
    _computed_names = frozenset({"total", "mean", "range"})

    def __init__(self, data):
        self._data = tuple(data)

    def _compute(self, name):
        data = object.__getattribute__(self, "_data")

        if name == "total":
            return sum(data)

        if not data:
            raise ValueError(f"cannot compute {name!r} for empty data")

        if name == "mean":
            return sum(data) / len(data)
        if name == "range":
            return max(data) - min(data)

        raise AssertionError("unexpected computed name")

    def __getattr__(self, name):
        computed_names = type(self)._computed_names
        if name not in computed_names:
            raise AttributeError(name)

        value = self._compute(name)
        self.__dict__[name] = value
        return value

    def replace_data(self, data):
        self._data = tuple(data)
        for name in self._computed_names:
            self.__dict__.pop(name, None)


stats = Stats([2, 4, 10])
assert stats.total == 16
assert stats.mean == 16 / 3
assert stats.range == 8

stats.replace_data([1, 1, 1, 1])
assert stats.total == 4
assert stats.mean == 1
assert stats.range == 0

empty = Stats([])
assert empty.total == 0
assert_raises(ValueError, getattr, empty, "mean")
assert_raises(AttributeError, getattr, empty, "median")

AttributeError('median')

# Problem 18 — Production-style integration challenge: layered configuration view

Build a configuration object with these layers, in priority order:

1. Real attributes/methods defined on the object/class.
2. Explicit user values from a mapping.
3. Deprecated aliases.
4. Default values.
5. Otherwise `AttributeError`.

Additional requirements:

- Deprecated aliases emit `DeprecationWarning`.
- Alias targets may come from user values or defaults.
- Dynamic public names should appear in `dir(obj)`.
- Internal storage must avoid accidental recursion.
- A real method name must not be shadowed by a mapping key of the same name.
- Provide a `snapshot()` method that returns the resolved public configuration.

In [36]:
# Starter
class ConfigViewStarter:
    def __init__(self, values, defaults, aliases=None):
        self._values = dict(values)
        self._defaults = dict(defaults)
        self._aliases = dict(aliases or {})

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

    def __dir__(self):
        # TODO
        return super().__dir__()

## Solution 18

Because the dynamic layers only need to run after ordinary lookup fails, `__getattr__` keeps the
implementation much safer than a full `__getattribute__` override.

The alias resolution explicitly calls `getattr(self, target)` because we intentionally want the full
resolution chain for the target.

In [37]:
import warnings

class ConfigView:
    def __init__(self, values, defaults, aliases=None):
        self._values = dict(values)
        self._defaults = dict(defaults)
        self._aliases = dict(aliases or {})

    def __getattr__(self, name):
        values = object.__getattribute__(self, "_values")
        if name in values:
            return values[name]

        aliases = object.__getattribute__(self, "_aliases")
        if name in aliases:
            target = aliases[name]
            warnings.warn(
                f"{name!r} is deprecated; use {target!r}",
                DeprecationWarning,
                stacklevel=2,
            )
            try:
                return getattr(self, target)
            except AttributeError:
                raise AttributeError(
                    f"deprecated alias {name!r} points to unavailable {target!r}"
                ) from None

        defaults = object.__getattribute__(self, "_defaults")
        if name in defaults:
            return defaults[name]

        raise AttributeError(f"{type(self).__name__} has no configuration field {name!r}")

    def __dir__(self):
        values = object.__getattribute__(self, "_values")
        defaults = object.__getattribute__(self, "_defaults")
        aliases = object.__getattribute__(self, "_aliases")
        dynamic = set(values) | set(defaults) | set(aliases)
        return sorted(set(super().__dir__()) | dynamic)

    def snapshot(self):
        values = object.__getattribute__(self, "_values")
        defaults = object.__getattribute__(self, "_defaults")
        names = set(defaults) | set(values)

        # Exclude dynamic names shadowed by real attributes/methods on the class hierarchy.
        def is_real_class_name(name):
            return any(name in base.__dict__ for base in type(self).__mro__)

        visible_dynamic_names = [name for name in names if not is_real_class_name(name)]
        return {name: getattr(self, name) for name in sorted(visible_dynamic_names)}


cfg = ConfigView(
    values={"host": "db.internal", "port": 6432, "snapshot": "must-not-shadow-method"},
    defaults={"host": "localhost", "port": 5432, "timeout": 30},
    aliases={"hostname": "host", "db_port": "port"},
)

assert cfg.host == "db.internal"
assert cfg.port == 6432
assert cfg.timeout == 30
assert callable(cfg.snapshot)  # real method wins over dynamic mapping key

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert cfg.hostname == "db.internal"
    assert cfg.db_port == 6432
    assert len(caught) == 2

assert {"host", "port", "timeout", "hostname", "db_port"} <= set(dir(cfg))

snap = cfg.snapshot()
assert snap["host"] == "db.internal"
assert snap["port"] == 6432
assert snap["timeout"] == 30
# Because snapshot is a real method, the mapping key named "snapshot" is intentionally inaccessible
# via normal dot-lookup. That is the declared precedence rule.

# Extra worked examples

The next cells are shorter examples designed to give you more code patterns to study and modify.

## Example A — Environment-style case-insensitive fallback

Normalize a missing public attribute name before looking it up in a backing dictionary.

In [38]:
class CaseInsensitiveView:
    def __init__(self, data):
        self._data = {str(k).lower(): v for k, v in data.items()}

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        data = object.__getattribute__(self, "_data")
        key = name.lower()
        try:
            return data[key]
        except KeyError:
            raise AttributeError(name) from None


env = CaseInsensitiveView({"API_URL": "https://example.test", "RETRIES": 3})
assert env.API_URL == "https://example.test"
assert env.api_url == "https://example.test"
assert env.ReTrIeS == 3

## Example B — Namespaced dynamic attributes

Support only names beginning with `feature_`, rejecting every other missing name.

In [39]:
class FeatureNamespace:
    def __init__(self, enabled):
        self._enabled = set(enabled)

    def __getattr__(self, name):
        prefix = "feature_"
        if name.startswith(prefix):
            feature = name[len(prefix):]
            return feature in object.__getattribute__(self, "_enabled")
        raise AttributeError(name)


features = FeatureNamespace({"search", "export"})
assert features.feature_search is True
assert features.feature_export is True
assert features.feature_payments is False
assert_raises(AttributeError, getattr, features, "search")

AttributeError('search')

## Example C — Memoized dynamic family of attributes

Interpret `square_12` as a dynamic request and cache the result.

In [40]:
class DynamicSquares:
    def __getattr__(self, name):
        prefix = "square_"
        if not name.startswith(prefix):
            raise AttributeError(name)

        suffix = name[len(prefix):]
        if not suffix.isdigit():
            raise AttributeError(name)

        n = int(suffix)
        value = n * n
        self.__dict__[name] = value
        return value


sq = DynamicSquares()
assert sq.square_12 == 144
assert sq.square_7 == 49
assert sq.__dict__["square_12"] == 144
assert_raises(AttributeError, getattr, sq, "square_x")

AttributeError('square_x')

## Example D — Read-only facade with delegated reads

A facade can delegate reads while controlling writes separately. This notebook focuses on read
accessors, so the example only shows the read side.

In [41]:
class ReadFacade:
    def __init__(self, target):
        object.__setattr__(self, "_target", target)

    def __getattr__(self, name):
        target = object.__getattribute__(self, "_target")
        return getattr(target, name)


class Payload:
    def __init__(self):
        self.x = 10
        self.y = 20

payload = Payload()
facade = ReadFacade(payload)
assert facade.x == 10
assert facade.y == 20

# Bonus problems — solve before opening the solutions

These are intentionally more compact. Each one has a full solution immediately below it.

## Bonus 1 — Alias cycle detection

Extend alias-based dynamic lookup so aliases may point to other aliases, but cycles such as
`a -> b -> c -> a` raise a clear error instead of recursing forever.

In [42]:
class AliasCycleStarter:
    def __init__(self, values, aliases):
        self._values = dict(values)
        self._aliases = dict(aliases)

    # TODO: implement cycle-safe resolution.

### Bonus 1 solution

In [43]:
class AliasResolver:
    def __init__(self, values, aliases):
        self._values = dict(values)
        self._aliases = dict(aliases)

    def _resolve_name(self, name):
        values = object.__getattribute__(self, "_values")
        aliases = object.__getattribute__(self, "_aliases")

        seen = []
        current = name

        while True:
            if current in values:
                return values[current]

            if current not in aliases:
                raise AttributeError(name)

            if current in seen:
                cycle = " -> ".join(seen + [current])
                raise RuntimeError(f"alias cycle detected: {cycle}")

            seen.append(current)
            current = aliases[current]

    def __getattr__(self, name):
        return self._resolve_name(name)


ar = AliasResolver(
    values={"canonical": 99},
    aliases={"old": "older", "older": "canonical"},
)
assert ar.old == 99

cycle = AliasResolver(values={}, aliases={"a": "b", "b": "c", "c": "a"})
exc = assert_raises(RuntimeError, getattr, cycle, "a")
assert "a -> b -> c -> a" in str(exc)

## Bonus 2 — A proxy that exposes selected target attributes only

Allow delegation only for names in an allowlist. All other missing names must raise `AttributeError`.

In [44]:
class AllowlistProxy:
    def __init__(self, target, allowed):
        self._target = target
        self._allowed = frozenset(allowed)

    def __getattr__(self, name):
        allowed = object.__getattribute__(self, "_allowed")
        if name not in allowed:
            raise AttributeError(name)
        target = object.__getattribute__(self, "_target")
        return getattr(target, name)


class Account:
    def __init__(self):
        self.name = "Ada"
        self.balance = 100
        self.secret = "hidden"


safe_account = AllowlistProxy(Account(), {"name", "balance"})
assert safe_account.name == "Ada"
assert safe_account.balance == 100
assert_raises(AttributeError, getattr, safe_account, "secret")

AttributeError('secret')

## Bonus 3 — Avoid accidentally swallowing target-side `AttributeError`

A subtle proxy issue: a target property may exist but raise `AttributeError` internally.
A broad `except AttributeError` in the proxy can mislabel that as "the target lacks the name."

Write a proxy that does not rewrite the exception; simply delegate and let the target's semantics pass
through unchanged.

In [45]:
class FragileTarget:
    @property
    def token(self):
        raise AttributeError("token source is unavailable")


class PassThroughProxy:
    def __init__(self, target):
        self._target = target

    def __getattr__(self, name):
        target = object.__getattribute__(self, "_target")
        return getattr(target, name)


ptp = PassThroughProxy(FragileTarget())
exc = assert_raises(AttributeError, getattr, ptp, "token")
assert "token source is unavailable" in str(exc)

## Bonus 4 — Per-name access limits with `__getattribute__`

Allow `secret` to be read only twice, then raise `AttributeError`, while all other attributes behave
normally.

In [46]:
class LimitedReads:
    def __init__(self, secret):
        self.secret = secret
        self._remaining = 2

    def __getattribute__(self, name):
        if name == "secret":
            remaining = object.__getattribute__(self, "_remaining")
            if remaining <= 0:
                raise AttributeError("secret read limit exceeded")
            object.__setattr__(self, "_remaining", remaining - 1)
        return object.__getattribute__(self, name)


lr = LimitedReads("abc")
assert lr.secret == "abc"
assert lr.secret == "abc"
assert_raises(AttributeError, getattr, lr, "secret")

AttributeError('secret read limit exceeded')

## Bonus 5 — Dynamic methods returned by `__getattr__`

For names like `multiply_by_7`, dynamically return a callable that multiplies its argument.
Cache the generated callable so the second read returns the same object.

In [47]:
class DynamicMultiplier:
    def __getattr__(self, name):
        prefix = "multiply_by_"
        if not name.startswith(prefix):
            raise AttributeError(name)

        text = name[len(prefix):]
        try:
            factor = int(text)
        except ValueError:
            raise AttributeError(name) from None

        def multiply(value, _factor=factor):
            return value * _factor

        multiply.__name__ = name
        self.__dict__[name] = multiply
        return multiply


dm = DynamicMultiplier()
f1 = dm.multiply_by_7
f2 = dm.multiply_by_7
assert f1 is f2
assert f1(6) == 42
assert getattr(dm, 'multiply_by_-3')(5) == -15

## Bonus 6 — Dynamic class attributes derived from the class name

Using a metaclass, expose `ClassName.type_label` dynamically as the lowercase class name.

In [48]:
class LabelMeta(type):
    def __getattr__(cls, name):
        if name == "type_label":
            return type.__getattribute__(cls, "__name__").lower()
        raise AttributeError(name)


class CustomerRecord(metaclass=LabelMeta):
    pass


class AuditEvent(metaclass=LabelMeta):
    pass


assert CustomerRecord.type_label == "customerrecord"
assert AuditEvent.type_label == "auditevent"

# Debugging clinic — identify the bug

Each snippet below shows a common failure mode. The corrected version follows.

## Bug 1 — Recursive internal read

Broken idea:

```python
def __getattribute__(self, name):
    self.log.append(name)
    return object.__getattribute__(self, name)
```

Why it fails: evaluating `self.log` itself calls `__getattribute__("log")`.

In [49]:
class SafeLogger:
    def __init__(self):
        self.log = []
        self.value = 5

    def __getattribute__(self, name):
        if name != "log":
            log = object.__getattribute__(self, "log")
            log.append(name)
        return object.__getattribute__(self, name)


sl = SafeLogger()
assert sl.value == 5
assert "value" in sl.log

## Bug 2 — Returning `None` for every missing attribute

Broken idea:

```python
def __getattr__(self, name):
    return None
```

This hides typos and causes `hasattr(obj, "typo")` to return `True`.

Correct approach: only return a fallback for names that are intentionally supported; otherwise raise
`AttributeError`.

In [50]:
class ExplicitFallbackOnly:
    def __getattr__(self, name):
        if name == "optional_note":
            return None
        raise AttributeError(name)


efo = ExplicitFallbackOnly()
assert efo.optional_note is None
assert hasattr(efo, "optional_note") is True
assert hasattr(efo, "typo") is False

## Bug 3 — Reimplementing Python's lookup using only `self.__dict__`

That misses class attributes, inherited attributes, properties, descriptors, and methods.
Prefer delegating to `object.__getattribute__` / `super().__getattribute__` rather than rebuilding the
lookup algorithm unless you have a very specialized reason.

In [51]:
class NormalLookupProof:
    class_attr = "class"

    @property
    def prop(self):
        return "property"

    def method(self):
        return "method"


nlp = NormalLookupProof()
assert "class_attr" not in nlp.__dict__
assert "prop" not in nlp.__dict__
assert "method" not in nlp.__dict__

assert nlp.class_attr == "class"
assert nlp.prop == "property"
assert nlp.method() == "method"

# Final challenge — Mini ORM-style field reader

Implement a small model abstraction where field values live in a private dictionary, while class
methods/properties still work normally.

Requirements:

- Declared fields are listed in a class-level `fields` set.
- `obj.field_name` reads from `_values`.
- Accessing a declared but unset field raises `AttributeError` with a helpful message.
- Undeclared names raise the ordinary-style `AttributeError`.
- Real methods must not be shadowed by field data.
- `dir(obj)` includes declared fields.
- Provide a `to_dict()` method.

In [52]:
# Starter
class ModelStarter:
    fields = frozenset()

    def __init__(self, **values):
        self._values = dict(values)

    def __getattr__(self, name):
        # TODO
        raise AttributeError(name)

    def __dir__(self):
        # TODO
        return super().__dir__()

## Final challenge solution

In [53]:
class Model:
    fields = frozenset()

    def __init__(self, **values):
        unknown = set(values) - set(type(self).fields)
        if unknown:
            raise TypeError(f"unknown field(s): {sorted(unknown)!r}")
        self._values = dict(values)

    def __getattr__(self, name):
        fields = type(self).fields
        if name not in fields:
            raise AttributeError(f"{type(self).__name__} has no attribute {name!r}")

        values = object.__getattribute__(self, "_values")
        try:
            return values[name]
        except KeyError:
            raise AttributeError(
                f"declared field {name!r} has not been set"
            ) from None

    def __dir__(self):
        return sorted(set(super().__dir__()) | set(type(self).fields))

    def to_dict(self):
        values = object.__getattribute__(self, "_values")
        return dict(values)


class User(Model):
    fields = frozenset({"name", "email", "age"})

    @property
    def display_name(self):
        return self.name.upper()

    def greeting(self):
        return f"Hello, {self.name}!"


u = User(name="Ada", email="ada@example.test")
assert u.name == "Ada"
assert u.email == "ada@example.test"
assert u.display_name == "ADA"
assert u.greeting() == "Hello, Ada!"
assert "age" in dir(u)
assert_raises(AttributeError, getattr, u, "age")
assert_raises(AttributeError, getattr, u, "unknown")
assert u.to_dict() == {"name": "Ada", "email": "ada@example.test"}
assert_raises(TypeError, User, name="Ada", bogus=1)

TypeError("unknown field(s): ['bogus']")

# Review checklist

Before shipping code that overrides attribute reads, verify all of these:

- [ ] Can normal attributes, class attributes, properties, descriptors, and methods still work?
- [ ] Does every truly missing name eventually raise `AttributeError`?
- [ ] Does `hasattr` behave correctly?
- [ ] Are internal reads inside `__getattribute__` delegated with `object.__getattribute__` or an
      equivalent safe mechanism?
- [ ] Have you tested missing private and dunder names?
- [ ] Have you tested method retrieval separately from method execution?
- [ ] If dynamic names exist, should `__dir__` expose them?
- [ ] If a proxy is involved, is attribute precedence explicit?
- [ ] Are special methods (`len`, iteration, arithmetic, context managers, etc.) handled separately if
      transparency is required?
- [ ] For class-level interception, is the hook on the metaclass?
- [ ] Do tests cover properties/descriptors that themselves raise exceptions?
- [ ] Are you avoiding attribute-hook tricks as a security mechanism?

# Suggested extensions

Try these after completing the notebook:

1. Add thread-safe lazy caching using a lock.
2. Build a proxy that forwards selected special methods explicitly.
3. Add a TTL to cached dynamic attributes.
4. Create a schema-backed config object that validates dynamic fields.
5. Add `__setattr__` and `__delattr__` to study the full attribute lifecycle.
6. Compare data descriptors, non-data descriptors, instance attributes, and `__getattr__` precedence.
7. Benchmark plain attribute access vs `__getattr__` fallback vs `__getattribute__` interception.
8. Add unit tests with `unittest` or `pytest`.